In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error


df = pd.read_parquet('train_prepared.parquet')

In [2]:
df.shape

(1460, 90)

In [3]:
X = df.drop('SalePrice', axis=1)

y = np.log1p(df['SalePrice'])

price_bins = pd.qcut(y, q=10, labels=False, duplicates='drop')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [4]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, price_bins)):
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    print(f'Fold {fold+1}: train={len(train_idx)}, val={len(val_idx)}, '
          f'mean_train={y_tr.mean():.3f}, mean_val={y_va.mean():.3f}')

Fold 1: train=1168, val=292, mean_train=12.026, mean_val=12.017
Fold 2: train=1168, val=292, mean_train=12.025, mean_val=12.021
Fold 3: train=1168, val=292, mean_train=12.021, mean_val=12.036
Fold 4: train=1168, val=292, mean_train=12.023, mean_val=12.030
Fold 5: train=1168, val=292, mean_train=12.026, mean_val=12.015


In [5]:
cat_cols = ['MSSubClass', 'MoSold', 'YrSold', 'Alley', 'MiscFeature',
            'MSZoning', 'Street', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2',
            'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd',
            'MasVnrType', 'Foundation', 'Heating', 'Electrical', 'GarageType',
            'SaleType', 'SaleCondition']

In [6]:
from catboost import CatBoostRegressor

rmse_scores = []
oof_preds = np.zeros(len(X))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, price_bins)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.05,
        depth=6,
        loss_function='RMSE',
        eval_metric='RMSE',
        random_seed=42,
        verbose=0,
        cat_features=cat_cols,
        early_stopping_rounds=100,
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True,
    )

    preds = model.predict(X_val)
    oof_preds[val_idx] = preds

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    rmse_scores.append(rmse)
    print(f'Fold {fold+1}: RMSE = {rmse:.4f}  (best_iter = {model.get_best_iteration()})')

print(f'\nСредний RMSE: {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}')

Fold 1: RMSE = 0.1097  (best_iter = 795)
Fold 2: RMSE = 0.1385  (best_iter = 1078)
Fold 3: RMSE = 0.1077  (best_iter = 513)
Fold 4: RMSE = 0.1058  (best_iter = 772)
Fold 5: RMSE = 0.1596  (best_iter = 350)

Средний RMSE: 0.1243 ± 0.0214
